# EMIT / Sentinel-2 pairing, alignment, and tiling

## 1. Setup

In [ ]:
import os, getpass
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

Cloning into 'HyperSpectralSuperResolution'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 360 (delta 8), reused 17 (delta 7), pack-reused 339 (from 1)
Receiving objects: 100% (360/360), 340.96 KiB | 13.64 MiB/s, done.
Resolving deltas: 100% (178/178), done.
/content/HyperSpectralSuperResolution


In [ ]:
!pip -q install -U py-tools-ds==0.24.1
!pip -q install -U arosics==1.13.2 geoarray==0.19.2
!pip install -q numpy xarray rasterio shapely pyproj matplotlib panel holoviews hvplot tqdm jupyter_bokeh pystac-client earthaccess netCDF4 spectral rioxarray hvplot POT
!pip install -q "git+https://github.com/EnSpec/hytools.git"
!apt-get install -y -q gdal-bin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.3/254.3 kB 17.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 29.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import importlib
import numpy as np
from py_tools_ds.geo.raster import reproject as reproj

importlib.reload(reproj)
_orig = reproj.warp_ndarray

def _warp_ndarray_mask_safe(ndarray, *args, **kwargs):
    for k in ("in_nodata", "out_nodata"):
        if k in kwargs and isinstance(kwargs[k], (bool, np.bool_)):
            kwargs[k] = int(kwargs[k])
    if isinstance(ndarray, np.ndarray) and ndarray.dtype == np.bool_:
        ndarray = ndarray.astype(np.uint8)
        kwargs.setdefault("in_nodata", 0)
        kwargs.setdefault("out_nodata", 0)
        kwargs.pop("out_dtype", None)
        kwargs.setdefault("rspAlg", "near")
    return _orig(ndarray, *args, **kwargs)

reproj.warp_ndarray = _warp_ndarray_mask_safe
print("warp_ndarray patched:", reproj.warp_ndarray is _warp_ndarray_mask_safe)

warp_ndarray patched: True


In [ ]:
import os, json, math, shutil, glob
from pathlib import Path
from datetime import datetime, date, timezone, timedelta
from pprint import pprint
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from shapely.geometry import Point, box
from tqdm import tqdm

import panel as pn
import holoviews as hv
import hvplot.xarray
from pystac_client import Client
import earthaccess as ea

pn.extension("tabulator")
hv.extension("bokeh")

# ── project modules ──────────────────────────────────────────────────────
from documentation.config import PipelineConfig

from documentation.pairs_artifacts import (
    # AOI catalogue & pipeline defaults
    load_aois_catalogue, save_aois_catalogue,
    get_aoi_by_name, get_aoi_by_index,
    load_pipeline_defaults, save_pipeline_defaults,
    aoi_slug, AoiPaths,
    # Pair-level
    make_pair_id, RunPaths, ReportWriter, TileRecord,
    load_pairs_sorted, write_aoi_pairs_csv, load_aoi_pairs_csv,
    pick_pair_by_rank,
    # Registry & run locks
    register_pair, registry_has_pair,
    save_run_lock, load_run_lock,
    # Metadata & manifests
    write_emit_metadata, write_s2_metadata,
    write_tile_metadata, write_manifest_csv, write_global_manifest,
    write_archive_map, tif_geo_summary,
    write_json, copy_any, describe_tif,
)

from documentation.report_builder import (
    ReportBuilder, R2Aggregator,
    plot_r2_histogram, plot_r2_per_band,
    plot_side_by_side_rgb, plot_example_tiles,
    plot_realignment_summary,
)

from data.pairing.pairs_utils import point_buffer_bbox
from data.pairing.pairing import pair_emit_to_s2_batched

from data.EMIT.emit_search import (
    search, download_reflectance,
    fetch_emit_umm_by_granuleur, refetch_emit_pick,
)
from data.EMIT.emit_utils import (
    load_emit_wavelengths_nm_from_nc, cache_wavelengths_json,
)
from data.S2.s2_search import (
    download_s2_spectral_stack, fetch_s2_item_by_id,
)

from geometry.alignment import align_emit_s2_pair
from geometry.EMIT_proj import convert_emit_nc_to_envi, crop_s2_stack_to_te
from geometry.crop_utils import (
    valid_bbox_in_map_coords, intersect_bounds,
    snap_bounds_to_grid, gdal_crop_projwin,
)
from geometry.arosics_coreg import (
    coregister_s2_granule_to_emit_granule, s2_bandmap_from_template,
)

from viz.plots import plot_s2_truecolor_from_stack, show_emit_rgb_from_envi

from tiles_helpers.utils import (
    find_valid_paired_tiles, save_tile_pair,
    plot_tile_pair_simple, write_emit_b32_tile,
)

from spectral.s2_to_emit import fit_tiles_batch, plot_r2_spectrum, scatter_band

print("All imports OK.")

All imports OK.


## 2. Authentication

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Login to https://urs.earthdata.nasa.gov/
ea.login(strategy="environment", persist=True)

## 3. Global parameters & output paths

In [ ]:
# ── Seed config files on first run ────────────────────────────────────────
DRIVE_ROOT = Path(f"/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22") # {datetime.now().strftime("%Y-%m-%d")}")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

for seed in ("aois.csv", "pipeline_defaults.json"):
    dst = DRIVE_ROOT / seed
    if not dst.exists():
        shutil.copy(seed, dst)
        print(f"Seeded {dst}")

print(load_aois_catalogue(DRIVE_ROOT).to_string(index=True))

                          name    lat     lon                                                      description                 land_cover
0             alps_switzerland  46.60    8.00                  Swiss Alps alpine meadow and rock — Switzerland            alpine/mountain
1                amazon_manaus  -2.50  -60.00                             Central Amazon basin — Manaus Brazil            tropical_forest
2              andes_altiplano -16.50  -68.50                      Altiplano high-altitude grassland — Bolivia           alpine/grassland
3              angola_woodland -12.00   18.00                                    Angolan miombo-savanna mosaic           woodland/savanna
4           appalachian_forest  36.50  -81.50               Southern Appalachian mixed hardwood — Virginia USA           temperate_forest
5                     aral_sea  45.00   58.50                   Aral Sea remnant basin — Kazakhstan-Uzbekistan             lake/salt_flat
6             argentina_pampas -35

In [ ]:
AOI_NAME  = None
AOI_INDEX = 29
PAIR_RANK = 0

_aoi = get_aoi_by_name(DRIVE_ROOT, AOI_NAME) if AOI_NAME else get_aoi_by_index(DRIVE_ROOT, AOI_INDEX)
_defaults = load_pipeline_defaults(DRIVE_ROOT)

config = PipelineConfig.from_dict({
    **_defaults,
    "aoi_lat":    _aoi["lat"],
    "aoi_lon":    _aoi["lon"],
    "drive_root": str(DRIVE_ROOT / datetime.now().strftime("%Y-%m-%d")),
})

print(f"\nAOI: {_aoi['name']}  ({_aoi['lat']}, {_aoi['lon']})  — {_aoi['land_cover']}")
print(f"Fingerprint: {config.run_fingerprint}")

ROOT = Path(config.local_root)
ROOT.mkdir(parents=True, exist_ok=True)

aoi = AoiPaths.build(DRIVE_ROOT, config.aoi_lat, config.aoi_lon)
config.to_json(aoi.config_json)
print(f"AOI slug: {aoi.slug}")


AOI: chaparral_california  (34.5, -118.5)  — mediterranean/shrubland
Fingerprint: 11db72806bcb
AOI slug: aoi_lat34.5_lon-118.5


## 4. Find cloud-free EMIT / S2 pairs

In [ ]:
aoi_geom = point_buffer_bbox(config.aoi_lon, config.aoi_lat, config.aoi_buffer_m)

search_start, search_end = config.search_bounds

picks = search(
    bbox=aoi_geom.bounds,
    start=search_start,
    end=search_end,
    cloud_cover=(0.0, 100.0),
)
print(f"{len(picks)} EMIT granules found in AOI.")

Found 21 granule(s).
21 EMIT granules found in AOI.


In [ ]:
for old in ROOT.glob("pairs_batch_*.jsonl"):
    old.unlink()

In [ ]:
run_summary = pair_emit_to_s2_batched(
    emit_items       = picks,
    aoi_geom_wgs84   = aoi_geom,
    out_dir          = ROOT,
    s2_api           = config.s2_api,
    s2_collection    = config.s2_collection,
    days             = config.pairing_days,
    window_days      = config.pairing_window_days,
    resume           = False,
    emit_top_n_per_day = config.emit_top_n_per_day,
    emit_max_cloud_pct = config.max_emit_cloud_pct,
    scl_cloud_max = config.max_s2_cloud_frac,
    top_k_prefilter      = config.top_k_prefilter,
    tile_m           = config.pairing_tile_m,
    sun_deg_max      = config.sun_deg_max,
    max_tod_diff_h   = config.max_tod_diff_h,
)

In [ ]:
import glob
from collections import Counter

base    = ROOT
reasons = Counter()
kept    = 0
total   = 0

pairs = []
for fn in sorted(glob.glob(f"{base}/pairs_batch_*.jsonl")):
    with open(fn) as f:
        for line in f:
            total += 1
            rec = json.loads(line)
            if rec.get("s2_id") and rec.get("s2_cloud_frac") is not None:
                kept += 1
                pairs.append(rec)
            else:
                dbg = rec.get("debug") or {}
                reasons[dbg.get("reason", "unknown")] += 1

print(f"Total candidates: {total}  |  Kept: {kept}")
print("\nRejection reasons:")
for k, v in reasons.most_common():
    print(f"  {k:<45s}  {v}")

Total candidates: 19  |  Kept: 2

Rejection reasons:
  no_candidates_after_time_sun_overlap           17


In [ ]:
pairs = load_pairs_sorted(ROOT, sort_by="expected_clear_km2")
write_aoi_pairs_csv(aoi, pairs, config_dict=config.to_dict())
print(f"\nPairs CSV: {aoi.pairs_csv}  ({len(pairs)} pairs ranked)")


Pairs CSV: /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/pairs.csv  (2 pairs ranked)


## 5. Download one pair

In [ ]:
pair = pick_pair_by_rank(pairs, rank=PAIR_RANK)

if registry_has_pair(aoi, pair["emit_granuleur"], pair["s2_id"],
                     config_fingerprint=config.run_fingerprint):
    print(f"Rank {PAIR_RANK} already completed with this config.")
else:
    print(f"Selected rank {PAIR_RANK}:")
    pprint(pair)

Selected rank 0:
{'debug': {'days': 3.0,
           'emit_dt': '2025-08-01T18:46:45+00:00',
           'max_tod_diff_h': 1.5,
           'n_candidates': 2,
           'n_prefilter': 2,
           'n_spatial_hits': 4,
           'picked': {'dt_diff_h': 24.026,
                      'expected_clear_km2': 6431.9936,
                      'meta_cloud': 0.207,
                      'overlap_h_m': 94393.11,
                      'overlap_km2': 6452.1225,
                      'overlap_w_m': 109798.0,
                      's2_id': 'S2C_11SLU_20250731_0_L2A',
                      's2_key': 'S2C_MSIL2A_20250731T182941_N0511_R027_T11SLU_20250731T230412.SAFE',
                      's2_updated': '2025-08-01T00:25:13.618000+00:00',
                      'scl_cloud': 0.00312,
                      'sun_term': 0.353},
           'sun_deg_max': 5.0,
           'tile_m': 6000.0,
           'top_candidates': [{'dt_diff_h': 24.026,
                               'expected_clear_km2': 6431.9936,
      

In [ ]:
emit_pick = refetch_emit_pick(pair["emit_granuleur"])
emit_pick

Collection: {'ShortName': 'EMITL2ARFL', 'Version': '001'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': -118.46862030029297, 'Latitude': 35.09526443481445}, {'Longitude': -119.25064849853516, 'Latitude': 34.47933578491211}, {'Longitude': -118.75711822509766, 'Latitude': 33.85270309448242}, {'Longitude': -117.97509002685547, 'Latitude': 34.468631744384766}, {'Longitude': -118.46862030029297, 'Latitude': 35.09526443481445}]}}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-08-01T18:46:45Z', 'EndingDateTime': '2025-08-01T18:46:56Z'}}
Size(MB): 3580.4495668411255
Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20250801T184645_2521312_005/EMIT_L2A_RFL_001_20250801T184645_2521312_005.nc', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20250801T184645_2521312_005/EMIT_L2A_RFLUNCERT_001_20250801T184645_2521312_005.nc', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20250801T184645_2521312_005/EMIT_L2A_MASK_001_20250801T184645_2521312_005.nc']

In [ ]:
s2_item  = fetch_s2_item_by_id(pair["s2_id"], stac_api=config.s2_api, collection=config.s2_collection)

emit_paths = download_reflectance(emit_pick, ROOT/"emit", assets=["_RFL_"], download_obs = True)
emit_nc    = Path(emit_paths[0])
s2_stack   = download_s2_spectral_stack(s2_item, ROOT/"s2")

print("EMIT:", emit_nc)
print("S2:  ", s2_stack)

Filtered to 1 reflectance-related asset link(s).


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Downloaded 1 OBS file(s) alongside reflectance.


S2C_11SLU_20250731_0_L2A_blue.tif: 100%|██████████| 225M/225M [00:09<00:00, 23.7MB/s]
S2C_11SLU_20250731_0_L2A_green.tif: 100%|██████████| 226M/226M [00:05<00:00, 41.7MB/s]
S2C_11SLU_20250731_0_L2A_red.tif: 100%|██████████| 231M/231M [00:04<00:00, 46.2MB/s]
S2C_11SLU_20250731_0_L2A_nir.tif: 100%|██████████| 223M/223M [00:13<00:00, 16.4MB/s]
S2C_11SLU_20250731_0_L2A_rededge1.tif: 100%|██████████| 60.3M/60.3M [00:01<00:00, 30.4MB/s]
S2C_11SLU_20250731_0_L2A_rededge2.tif: 100%|██████████| 60.1M/60.1M [00:01<00:00, 40.0MB/s]
S2C_11SLU_20250731_0_L2A_rededge3.tif: 100%|██████████| 60.4M/60.4M [00:02<00:00, 26.2MB/s]
S2C_11SLU_20250731_0_L2A_swir16.tif: 100%|██████████| 60.3M/60.3M [00:01<00:00, 34.6MB/s]
S2C_11SLU_20250731_0_L2A_swir22.tif: 100%|██████████| 60.3M/60.3M [00:01<00:00, 38.3MB/s]
S2C_11SLU_20250731_0_L2A_nir08.tif: 100%|██████████| 60.4M/60.4M [00:01<00:00, 33.8MB/s]
S2C_11SLU_20250731_0_L2A_SCL.tif: 100%|██████████| 2.31M/2.31M [00:00<00:00, 6.31MB/s]


  Flagged 2 pixels; inpainted 2 band-pixels.
EMIT: pairs_output/emit/EMIT_L2A_RFL_001_20250801T184645_2521312_005.nc
S2:   pairs_output/s2/S2C_11SLU_20250731_0_L2A_S2_10band_10m.tif


In [ ]:
obs_path = emit_paths[-1] if len(emit_paths) > 1 else None

## 6. Preprocessing — metadata, paths, report

In [ ]:
wl_nm   = load_emit_wavelengths_nm_from_nc(str(emit_nc))
wl_json = Path(emit_nc).with_suffix(".wavelengths.json")
cache_wavelengths_json(wl_nm, wl_json)
print(f"{len(wl_nm)} EMIT bands — {wl_nm[0]:.1f}-{wl_nm[-1]:.1f} nm")

285 EMIT bands — 380.9-2492.8 nm


In [ ]:
s2_item.to_dict()

{'type': 'Feature',
 'stac_version': '1.1.0',
 'stac_extensions': ['https://stac-extensions.github.io/view/v1.0.0/schema.json',
  'https://stac-extensions.github.io/mgrs/v1.0.0/schema.json',
  'https://stac-extensions.github.io/sentinel-2/v1.0.0/schema.json',
  'https://stac-extensions.github.io/grid/v1.1.0/schema.json',
  'https://stac-extensions.github.io/projection/v2.0.0/schema.json',
  'https://stac-extensions.github.io/raster/v1.1.0/schema.json',
  'https://stac-extensions.github.io/processing/v1.1.0/schema.json',
  'https://stac-extensions.github.io/eo/v1.1.0/schema.json'],
 'id': 'S2C_11SLU_20250731_0_L2A',
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-119.19753042986382, 35.22311612951892],
    [-119.17148688046915, 34.23369657781449],
    [-117.97960291503624, 34.24901663469657],
    [-117.99136083035947, 35.23900847829063],
    [-119.19753042986382, 35.22311612951892]]]},
 'bbox': [-119.19753, 34.233697, -117.979603, 35.239008],
 'properties': {'created': '2025-08-01

In [ ]:
# Generate pair_id and build per-pair folder structure
pair_id = make_pair_id(emit_nc, s2_item.to_dict())
paths = RunPaths.build(
    emit_nc=emit_nc,
    local_root=ROOT / pair_id,
    aoi=aoi,
    pair_id=pair_id,
    s2_item=s2_item,
)

run_entry = register_pair(
    aoi, pair=pair, pair_id=pair_id,
    config_fingerprint=config.run_fingerprint,
    status="started",
)

report = ReportBuilder(
    paths.drive_report_md,
    html_path=paths.drive_report_md.with_suffix(".html"),
    mode="overwrite",
)
report.start(title=f"EMIT / S2 pair: {paths.pair_id}")

report.add_config_table(config)
config.to_json(paths.drive_meta / "pipeline_config.json")

emit_summary = write_emit_metadata(emit_pick, paths.drive_meta, report=report)
s2_summary   = write_s2_metadata(s2_item,   paths.drive_meta, report=report)

In [ ]:
bounds_wgs84   = emit_summary["spatial"]["bounds_wgs84"]
centroid_wgs84 = emit_summary["spatial"]["centroid_wgs84"]
print("EMIT centroid:", centroid_wgs84)
print("S2 id:        ", s2_summary["id"])

EMIT centroid: {'lon': -118.61286926269533, 'lat': 34.47398376464844}
S2 id:         S2C_11SLU_20250731_0_L2A


## 7. Convert, crop, co-register, and trim

In [ ]:
alignment = align_emit_s2_pair(
    emit_nc = emit_nc,
    s2_stack = s2_stack,
    out_dir = paths.local_alignment,
    config = config,
    wl_nm = wl_nm,
    emit_obs_nc = obs_path,
    report = report,
    keep_intermediate = False,
    emit_info_save_path = paths.drive_alignment / "emit_conversion.json",
)

if not alignment.success:
    raise RuntimeError(f"Co-registration failed: {alignment.coreg_info}")

envi_bin_trimmed    = alignment.emit_envi_trimmed
out_s2_tif_trimmed  = alignment.s2_tif_trimmed
emit_conv_info      = alignment.emit_conv_info
envi_bin            = alignment.emit_envi_full

print(f"EMIT: {envi_bin_trimmed}")
print(f"S2:   {out_s2_tif_trimmed}")

pairs_output/emit/EMIT_L2A_RFL_001_20250801T184645_2521312_005.nc
Opened with: netCDF4
Disabling auto mask/scale for variable 'reflectance'
[DEBUG] data.dimensions=('downtrack', 'crosstrack', 'bands') -> transpose_raw_yx=False
Exporting EMIT L2A_RFL dataset
[DEM-CORR] Downloading Copernicus GLO-30 ...
  [DEM] downloading COP30_N33_W120.tif ...
  [DEM] downloading COP30_N33_W119.tif ...
  [DEM] downloading COP30_N33_W118.tif ...
  [DEM] downloading COP30_N34_W120.tif ...
  [DEM] downloading COP30_N34_W119.tif ...
  [DEM] downloading COP30_N34_W118.tif ...
  [DEM] 6 downloaded, 0 cached, 6 tiles total → /tmp/dem_cache/cop30_mosaic.vrt
  [DEM-CORR-RAW] Sampling DEM at raw pixel locations ...
  [DEM] geoid undulation applied to sampled points: mean N=-33.34 m
  [DEM] sampled 1589760/1589760 points from /tmp/dem_cache/cop30_mosaic.vrt
  [DEM] elev range: -36.3 – 2217.8 m
  [DEM-CORR-RAW] dh: mean=-2.28 m  max_abs=136.20 m
  [DEM-CORR-RAW] dlon: mean=-0.000003°  dlat: mean=-0.000001°
  [VRT]

Polygonize progress     |============================================------| 87.0% Complete  => 0:00:00

Calculating footprint polygon and actual data corner coordinates for reference image...


Polygonize progress     |==================================================| 100.0% Complete  => 0:00:00


Bounding box of calculated footprint for reference image:
	(300000.0, 3790200.0, 409800.0, 3872820.0)
Calculating footprint polygon and actual data corner coordinates for image to be shifted...


...

Calculating tie point grid (600 points) using 8 CPU cores...


	progress: |==================================================| 100.0% Complete  => 0:01:32


Found 584 matches.
Performing validity checks...
18 tie points flagged by level 1 filtering (reliability).
0 tie points flagged by level 2 filtering (SSIM).
57 tie points flagged by level 3 filtering (RANSAC)
512 valid tie points remain after filtering.
Correcting geometric shifts...


Translating progress |==================================================| 100.0% Complete  => 0:00:03
Warping progress     |==================================================| 100.0% Complete  => 0:00:52


Writing GeoArray of size (8376, 10980, 10) to pairs_output/20250801T184645_T11SLU_20250731/alignment/s2/S2C_11SLU_20250731_0_L2A_coreg.tif.
EMIT: pairs_output/20250801T184645_T11SLU_20250731/alignment/emit_utm/L2A_RFL_001_20250801T184645_2521312_005.bin
S2:   pairs_output/20250801T184645_T11SLU_20250731/alignment/s2/S2C_11SLU_20250731_0_L2A_coreg.tif


## 8. Visualisation

In [ ]:
# Before alignment (use untrimmed ENVI + the S2 overlap crop)
rgb_before = plot_side_by_side_rgb(
    s2_path=alignment.s2_overlap_path or s2_stack,
    emit_path=envi_bin,
    out_path=paths.drive_plots / "comparison_before_coreg.png",
    wl_nm=wl_nm,
    title="Before co-registration",
)
report.add_image("S2 vs EMIT — Before Co-registration", rgb_before)

# After alignment
rgb_after = plot_side_by_side_rgb(
    s2_path=out_s2_tif_trimmed,
    emit_path=envi_bin_trimmed,
    out_path=paths.drive_plots / "comparison_after_coreg.png",
    wl_nm=wl_nm,
    title="After co-registration + trim",
)
report.add_image("S2 vs EMIT — After Co-registration", rgb_after)

Reading: L2A_RFL_001_20250801T184645_2521312_005.bin
Picked band 34 (0-based) at 633.7 nm for target 630.0 nm
Picked band 20 (0-based) at 529.4 nm for target 532.0 nm
Picked band 11 (0-based) at 462.5 nm for target 465.0 nm
Reading: L2A_RFL_001_20250801T184645_2521312_005.bin
Picked band 34 (0-based) at 633.7 nm for target 630.0 nm
Picked band 20 (0-based) at 529.4 nm for target 532.0 nm
Picked band 11 (0-based) at 462.5 nm for target 465.0 nm


## 9. Tiling and export

In [ ]:
valid_tiles = find_valid_paired_tiles(
    emit_path = str(envi_bin_trimmed),
    s2_path   = str(out_s2_tif_trimmed),
    **config.tile_params,
)
print(f"{len(valid_tiles)} valid tile positions found.")

120 valid tile positions found.


In [ ]:
from rasterio.windows import from_bounds, transform as window_transform


In [ ]:
tile_records: list[TileRecord] = []
emit_b32_idx: np.ndarray | None = None

for tile_info in tqdm(valid_tiles):
    k = int(tile_info["idx"])

    # pair_id is included in tile filenames; realign refines S2→EMIT alignment per tile
    emit_out, s2_out, realign_stats = save_tile_pair(
        emit_path = str(envi_bin_trimmed),
        s2_path   = str(out_s2_tif_trimmed),
        tile_info = tile_info,
        out_dir   = paths.drive_tiles,
        pair_id   = paths.pair_id,
        emit_wavelengths_nm = wl_nm,
        realign   = True,                  # ← per-tile sub-pixel alignment
        realign_max_shift = 1.0,           # reject shifts > 1 EMIT pixel
    )

    emit_out_b32, emit_b32_idx = write_emit_b32_tile(
        emit_out,
        num_keep   = 32,
        idx_0based = emit_b32_idx,
        overwrite  = True,
        wavelengths_nm=wl_nm
    )

    plot_png = paths.drive_plots / f"{paths.pair_id}_tile{k:03d}_pair.png"
    plot_tile_pair_simple(
        emit_tile_path = str(emit_out),
        s2_tile_path   = str(s2_out),
        title_suffix   = f"tile {k:03d}",
        save_path      = str(plot_png),
        show           = False,
        emit_wavelengths_nm = wl_nm,
    )

    rec = TileRecord(
        idx             = k,
        emit_tif        = str(emit_out),
        s2_tif          = str(s2_out),
        pair_id         = paths.pair_id,
        plot_png        = str(plot_png),
        emit_black_frac = tile_info.get("emit_black_frac"),
        s2_black_frac   = tile_info.get("s2_black_frac"),
    )
    rec.emit_geo = tif_geo_summary(emit_out)
    rec.s2_geo   = tif_geo_summary(s2_out)
    if realign_stats is not None:
        rec.realign_stats = realign_stats  # shift_emit_px, shift_s2_px, applied
    rec.emit_b32_tif = str(emit_out_b32)
    rec.emit_b32_indices_0based = emit_b32_idx.tolist()

    write_tile_metadata(
        rec,
        tile_info    = tile_info,
        out_dir      = paths.drive_tile_meta,
        emit_granule = emit_summary.get("granule_ur"),
        emit_time    = emit_summary.get("time"),
        s2_id        = s2_summary.get("id"),
        s2_datetime  = s2_summary.get("datetime"),
        params       = config.tile_params,
    )
    tile_records.append(rec)

write_manifest_csv(paths.drive_manifest_csv, tile_records)

# ── Realignment report (after tiling, before fitting) ─────────────────

100%|██████████| 120/120 [03:58<00:00,  1.99s/it]


PosixPath('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/manifest.csv')

In [ ]:
realign_plot = plot_realignment_summary(
    tile_records,
    paths.drive_plots / "realignment_shifts.png",
    title=f"Realignment — {paths.pair_id}",
)
report.add_realignment_section(tile_records, plot_path=realign_plot)

In [ ]:
global_csv = write_global_manifest(aoi)
print(f"Global manifest: {global_csv}  ({pd.read_csv(global_csv).shape[0]} tiles total)")

Global manifest: /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/global_manifest.csv  (120 tiles total)


In [ ]:
example_centroid = None
if tile_records and isinstance(tile_records[0].s2_geo, dict):
    example_centroid = tile_records[0].s2_geo.get("centroid_wgs84")

report.section("Tiles summary", [
    f"Number of saved tiles: {len(tile_records)}",
    f"Example tile centroid (S2, WGS84): {example_centroid}" if example_centroid else None,
    f"Tile metadata folder: {paths.drive_tile_meta}",
    f"Manifest: {paths.drive_manifest_csv}",
])

In [ ]:
from spectral.s2_to_emit import fit_tile, plot_r2_spectrum, plot_spectral_match

synth_dir = paths.drive_regression_synth
synth_dir.mkdir(parents=True, exist_ok=True)

r2_agg = R2Aggregator(wavelengths_nm=wl_nm)
matchers = {}
wl_b32 = wl_nm[rec.emit_b32_indices_0based]


for rec in tqdm(tile_records, desc="Fitting + applying"):
    if rec.emit_b32_tif is None:
        continue

    try:
        matcher, stats = fit_tile(
            s2_tile_path = rec.s2_tif,
            emit_b32_tile_path = rec.emit_b32_tif,
            **config.spectral_params,
            verbose=False,
        )
    except ValueError as e:
        print(f"  tile {rec.idx:03d}  SKIPPED — {e}")
        continue

    matchers[rec.idx] = matcher
    matcher.save(paths.drive_regression_matchers / f"tile_{rec.idx:03d}_matcher.joblib")

    # R^2
    r2_agg.add(rec.idx, np.array(stats["r2_per_band"]), stats["r2_mean"])

    out_path = synth_dir / Path(rec.s2_tif).name.replace("_s2.tif", "_regression_synth.tif")
    matcher.apply_to_tile(rec.s2_tif, out_path=out_path)

    print(f"  tile {rec.idx:03d}  R² mean={stats['r2_mean']:.4f}")

    synth_plot = paths.drive_plots / f"{paths.pair_id}_tile{rec.idx:03d}_synth.png"
    plot_spectral_match(
        rec.s2_tif,
        str(out_path),
        rec.emit_b32_tif,
        title_suffix=f"tile {rec.idx:03d}",
        r2_mean=stats["r2_mean"],
        save_path=str(paths.drive_plots / f"{paths.pair_id}_tile{rec.idx:03d}_synth.png"),
        show=True,
        wavelengths_nm = wl_b32
    )

print(f"\nDone. {len(matchers)} matchers fitted.")

Fitting + applying:   0%|          | 0/120 [00:00<?, ?it/s]

Regressor saved → /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/regression_matchers/tile_026_matcher.joblib
  tile 026  R² mean=0.8725


Fitting + applying:   1%|          | 1/120 [00:06<11:56,  6.02s/it]

Regressor saved → /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/regression_matchers/tile_043_matcher.joblib
  tile 043  R² mean=0.8356


Fitting + applying:   2%|▏         | 2/120 [00:12<12:50,  6.53s/it]

...

Regressor saved → /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/regression_matchers/tile_228_matcher.joblib
  tile 228  R² mean=0.6635


Fitting + applying:  99%|█████████▉| 119/120 [13:29<00:07,  7.16s/it]

Regressor saved → /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/regression_matchers/tile_229_matcher.joblib
  tile 229  R² mean=0.6604


Fitting + applying: 100%|██████████| 120/120 [13:35<00:00,  6.79s/it]


Done. 120 matchers fitted.


In [ ]:
# ── R² summary and histograms ────────────────────────────────────────────
r2_summary = r2_agg.summary()
print(f"R² across {r2_summary['n_tiles']} tiles:")
print(f"  Mean:   {r2_summary['global_mean']:.4f}")
print(f"  Median: {r2_summary['global_median']:.4f}")
print(f"  Range:  [{r2_summary['global_min']:.4f}, {r2_summary['global_max']:.4f}]")
print(f"  Std:    {r2_summary['global_std']:.4f}")

# Histogram of mean R² per tile
hist_path = plot_r2_histogram(
    r2_agg.r2_mean,
    paths.drive_plots / "r2_histogram.png",
    title=f"R² distribution — {paths.run_id}",
)

# Per-band R² box plot
band_path = plot_r2_per_band(
    r2_agg.r2_per_band,
    r2_agg.wavelengths_nm,
    paths.drive_plots / "r2_per_band.png",
)

# Example tile images
example_pngs = plot_example_tiles(tile_records, paths.drive_plots, n=3, wl_nm=wl_nm)

# Add everything to report
report.add_r2_section(r2_agg, histogram_path=hist_path, per_band_path=band_path)
for png in example_pngs:
    report.add_image(f"Example tile", png)

R² across 120 tiles:
  Mean:   0.7415
  Median: 0.7494
  Range:  [0.5656, 0.8919]
  Std:    0.0813


## 10. Archive to Drive and finalise report

In [ ]:
copy_any(
    paths.local_alignment,
    paths.drive_alignment,
    overwrite=True,
    exclude=["tmp", "*.aux.xml", "*.ovr"],
)
copy_any(emit_nc,  paths.drive_emit / Path(emit_nc).name,  overwrite=True)
copy_any(s2_stack, paths.drive_s2   / Path(s2_stack).name, overwrite=True)

archive_map = {
    "local_ROOT":                str(ROOT),
    "drive_pair_folder":         str(DRIVE_ROOT),
    "drive_raw_emit":            str(paths.drive_emit),
    "drive_raw_s2":              str(paths.drive_s2),
    "drive_emit_reprojections":  str(paths.drive_alignment),
}
write_archive_map(paths.drive_meta / "archive_map.json", archive_map, report=report)
print("Archive complete.  Report:", paths.drive_report_md)

Archive complete.  Report: /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/report.md


In [ ]:
html_path = report.finalise_html(title=f"Run {paths.run_id}")
print(f"Report (md):   {report.path}")
print(f"Report (html): {html_path}")

Report (md):   /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/report.md
Report (html): /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/report.html


In [ ]:
register_pair(
    aoi, pair=pair, pair_id=pair_id,
    run_uid=run_entry["run_uid"],
    config_fingerprint=config.run_fingerprint,
    status="completed",
)

save_run_lock(
    paths.drive_root, pair=pair, pair_id=pair_id,
    run_uid=run_entry["run_uid"],
    config_fingerprint=config.run_fingerprint,
    config_dict=config.to_dict(),
)

PosixPath('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22/aoi_lat34.5_lon-118.5/20250801T184645_T11SLU_20250731/run_lock.json')

In [ ]:
!rm -fr /content/HyperSpectralSuperResolution/pairs_output/*

In [ ]:
# paths.drive_alignment / "emit_utm"

In [ ]:
# from spectral_diagnosis import run_spectral_diagnosis


# report = run_spectral_diagnosis(
#     paths.drive_tiles,
#     wl_nm=wl_nm,                          # full 285-band wavelengths
#     emit_source_path=envi_bin_trimmed,  # pre-tiling ENVI file
#     max_tiles=30,                          # speed up by sampling
#     save_dir=paths.drive_plots,
# )

In [ ]:
# plot_png = paths.drive_plots / f"example_CNMF.png"
# emit_out = paths.drive_tiles / "20250422T182135_T11SPS_20250419_tile041_emit_b32.tif"
# s2_out = paths.drive_tiles / "20250422T182135_T11SPS_20250419_tile041_s2.tif"
# synth = "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/1773516503.805082/20250422T182135_T11SPS_20250419/CNMF/20250422T182135_T11SPS_20250419_tile041_CNMF_fused.tif"
# plot_tile_pair_simple(
#         emit_tile_path = str(emit_out),
#         s2_tile_path   = str(s2_out),
#         title_suffix   = f"tile {k:03d}",
#         save_path      = str(plot_png),
#         show           = True,
#         emit_wavelengths_nm = wl_nm,
#     )


In [ ]:
# paths.drive_root

In [ ]:
# CNMF_dir = paths.drive_root / "CNMF"
# for rec in tqdm(tile_records, desc="Plotting CNMF results..."):
#     if rec.emit_b32_tif is None:
#         continue



#     synth_path = CNMF_dir / Path(rec.s2_tif).name.replace("_s2.tif", "_CNMF_fused.tif")

#     synth_plot = paths.drive_plots / f"{paths.pair_id}_tile{rec.idx:03d}_synth.png"
#     plot_spectral_match(
#         rec.s2_tif,
#         str(synth_path),
#         rec.emit_b32_tif,
#         title_suffix=f"tile {rec.idx:03d}",
#         # r2_mean=stats["r2_mean"],
#         save_path=str(paths.drive_plots / f"{paths.pair_id}_tile{rec.idx:03d}_CNMF_synth.png"),
#         show=True,
#         wavelengths_nm = wl_b32
#     )

# print(f"\nDone. {len(matchers)} matchers fitted.")